# Verify Cluster Deployment

End-to-end verification of the custom deep research system running on OpenShift AI with Kagenti.

## 1. Load Environment

In [ ]:
import os
import subprocess
import json
import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

NAMESPACE = os.getenv("NAMESPACE", "doc-research-lab")
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN", "")
print(f"Cluster: {CLUSTER_DOMAIN}")
print("✅ Environment loaded")

## 2. Check Pod Health

In [ ]:
r = subprocess.run(
    ["oc", "get", "pods", "-n", NAMESPACE,
     "-o", "jsonpath={range .items[*]}{.metadata.name}{'\t'}{.status.phase}{'\n'}{end}"],
    capture_output=True, text=True,
)

if r.returncode == 0:
    pods = r.stdout.strip().split("\n")
    all_running = True
    for pod in pods:
        if pod:
            name, phase = pod.split("\t")
            status = "✅" if phase == "Running" else "❌"
            print(f"  {status} {name}: {phase}")
            if phase != "Running":
                all_running = False
    if all_running:
        print("\n✅ All pods running")
    else:
        print("\n⚠️ Some pods not ready")
else:
    print(f"❌ Error: {r.stderr}")

## 3. Test A2A Discovery via Routes

In [ ]:
# Get routes for each agent
r = subprocess.run(
    ["oc", "get", "routes", "-n", NAMESPACE, "-o", "json"],
    capture_output=True, text=True,
)

if r.returncode == 0:
    routes = json.loads(r.stdout)
    agent_routes = {}
    for route in routes.get("items", []):
        name = route["metadata"]["name"]
        host = route["spec"]["host"]
        agent_routes[name] = f"https://{host}"
        print(f"  Route: {name} → https://{host}")

    # Test AgentCard discovery
    print("\nTesting A2A discovery:")
    for name, url in agent_routes.items():
        try:
            resp = httpx.get(f"{url}/.well-known/agent-card.json", timeout=10, verify=False)
            if resp.status_code == 200:
                card = resp.json()
                print(f"  ✅ {name}: {card.get('name', 'N/A')}")
            else:
                print(f"  ❌ {name}: HTTP {resp.status_code}")
        except Exception as e:
            print(f"  ❌ {name}: {e}")
else:
    print("⚠️ No routes found. Agents may use internal Services only.")

## 4. End-to-End Research Test on Cluster

In [ ]:
import time

# Use the orchestrator route or port-forward
orchestrator_url = agent_routes.get("orchestrator", "http://localhost:8100")

payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": "cluster-test-001",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "Summarize the key findings from the indexed documents."}],
        }
    },
}

print(f"Testing orchestrator at: {orchestrator_url}")
start = time.time()

try:
    resp = httpx.post(orchestrator_url, json=payload, timeout=120, verify=False)
    elapsed = time.time() - start
    result = resp.json()

    if "result" in result:
        print(f"\nResponse received in {elapsed:.1f}s")
        print(json.dumps(result["result"], indent=2)[:1000])
        print("\n✅ Cluster deployment verified — multi-agent pipeline working")
    elif "error" in result:
        print(f"\n❌ A2A error: {result['error']}")

except Exception as e:
    print(f"❌ Error: {e}")
    print("   Try: oc port-forward svc/orchestrator 8100:8100 -n", NAMESPACE)